# DDS + CIC Calibration and Linearity Analysis

This notebook explains how to measure, analyze, and calibrate the output of an ADC followed by a Polyphase Filter Bank (PFB) and a DDS + CIC block. 

## 1. System Overview

We consider the following chain:

1. RF signal entering an ADC:
   - Frequency: $f_{RF} = 300\,\mathrm{MHz}$
   - Power: variable $P_{in}$ (dBm)
   - ADC sampling rate: $F_s = 4096\,\mathrm{MHz}$
2. Digital decimation by 2 and mixing by $-F_s/4$ to create a complex baseband.
3. Polyphase Filter Bank (PFB) with 8 channels and decimation factor 8:
   - Output bandwidth per channel: 256 MHz
   - PFB **net gain** = polyphase accumulation gain + qout attenuation:
     $$G_{\text{PFB,net}} = +20\log_{10}(D) - 20\log_{10}(2^{q_{\text{out}}}) = +18.06 - 36.12 = -18.06\,\text{dB}$$
   - With `qout = 6` and `D = 8` (see Notebook 02)
4. DDS to translate PFB output signal to DC
5. CIC for decimating the signal
   - Decimation factor R = 8
   - Differential delay M = 1
   - Order N = 3
   - CIC DC gain: $G_{\text{CIC}} = (RM)^N = 512 = +54.19\,\text{dB}$
   - DDS + CIC block output quantization controlled by `qprod(14)`

The goal is to measure the tone level at the output of the DDS + CIC and convert it to real RF power in dBm.

---


---

## Gain Compensation Between PFB and CIC

In the processing chain

```
PFB → DDS → CIC
```

the CIC decimator introduces a significant gain that must be taken into account to avoid overflow and to maintain a convenient signal scaling.

For a CIC filter with:

* Decimation factor (R = 8)
* Differential delay (M = 1)
* Order (N = 3)

the DC gain of the filter is

$$
G_{CIC} = (RM)^N = (8 \cdot 1)^3 = 512 \quad (+54.19\,\text{dB})
$$

---

### PFB Net Gain (corrected model)

As established in [PFB data analysis](./02_PFB_data.ipynb), the PFB introduces **two** amplitude effects:

1. **Polyphase accumulation gain:** the D = 8 polyphase branches coherently accumulate, producing an amplitude gain of D:
$$G_{\text{polyphase}} = +20\log_{10}(D) = +20\log_{10}(8) = +18.06\,\text{dB}$$

2. **qout right-shift attenuation:** the FFT output is right-shifted by `qout = 6` bits before truncation to 16 bits:
$$G_{q_{\text{out}}} = -20\log_{10}(2^{q_{\text{out}}}) = -20\log_{10}(64) = -36.12\,\text{dB}$$

The **net PFB gain** is therefore:
$$G_{\text{PFB,net}} = G_{\text{polyphase}} + G_{q_{\text{out}}} = +18.06 - 36.12 = -18.06\,\text{dB}$$

> **Common mistake:** using only the qout term ($-36.12\,\text{dB}$) without the polyphase gain ($+18.06\,\text{dB}$) produces a systematic calibration error of ~20 dB.

---

### Compensation Using qout

To pre-compensate the CIC gain at the PFB output (keeping the signal level roughly constant across the full chain), one can choose `qout` so that the combined PFB–CIC gain is near unity.

The condition is:

$$G_{\text{polyphase}} + G_{q_{\text{out}}} + G_{\text{CIC}} \approx 0\,\text{dB}$$

$$+18.06 - 6.02 \cdot q_{\text{out}} + 54.19 = 0$$

$$q_{\text{out}} = \frac{18.06 + 54.19}{6.02} \approx 12$$

In practice, the firmware saturates `QOUT_REG` at 11 (hardware protection), so perfect compensation is not achievable with `qout` alone. For this experiment `qout = 6` is used, meaning the CIC gain is **not** pre-compensated; instead the full gain budget is tracked analytically in the calibration (see Section 3).

---


## Quantization Scaling at the CIC Output

After the CIC filter, the signal is reduced from 24 bits to 16 bits using the `qdata` module.

The module selects a 16-bit slice of the input word according to

```verilog
qsel_i = BIN - QSEL_REG - 1
dout   = din[qsel_i -: BOUT]
```

For the configuration used in this experiment:

```
BIN      = 24
BOUT     = 16
QSEL_REG = 14
```

This effectively corresponds to a right shift of the input signal by $\text{QSEL}_{\text{REG}}$ bits, resulting in an amplitude scaling:

$$G_{\text{QDATA}} = 2^{-\text{QSEL}_{\text{REG}}} = 2^{-14} = \frac{1}{16384} \quad (-84.29\,\text{dB})$$

---

## Total Gain of the DSP Chain

The signal processing chain from PFB output to DDS+CIC output is:

```
PFB  →  DDS  →  CIC  →  qdata
```

Each stage contributes a gain factor.

### PFB Net Gain (polyphase accumulation + qout)

$$G_{\text{PFB,net}} = +20\log_{10}(D) - 20\log_{10}(2^{q_{\text{out}}})$$

With $D = 8$ and $q_{\text{out}} = 6$:

$$G_{\text{PFB,net}} = +18.06 - 36.12 = -18.06\,\text{dB}$$

### CIC Gain

For a CIC decimator with R = 8, M = 1, N = 3:

$$G_{\text{CIC}} = (RM)^N = (8 \cdot 1)^3 = 512 \quad (+54.19\,\text{dB})$$

### Quantization Scaling (qdata)

$$G_{\text{QDATA}} = 2^{-14} \quad (-84.29\,\text{dB})$$

---

## Combined Gain from PFB output to DDS+CIC output

$$G_{\text{DDS+CIC\ block}} = G_{\text{CIC}} \cdot G_{\text{QDATA}}$$

$$G_{\text{DDS+CIC\ block}} = 2^{9} \cdot 2^{-14} = 2^{-5} \quad (-30.10\,\text{dB})$$

## Total Chain Gain (from PFB output to final output)

$$G_{\text{TOTAL}} = G_{\text{PFB,net}} \cdot G_{\text{CIC}} \cdot G_{\text{QDATA}}$$

$$G_{\text{TOTAL}} = 2^{-18.06/6.02} \cdot 2^{9} \cdot 2^{-14} = 2^{-3} \cdot 2^{9} \cdot 2^{-14} = 2^{-8} \quad (-48.16\,\text{dB})$$

Since $G_{\text{PFB,net}} = -18.06\,\text{dB}$ is not an exact power of two, the exact linear total is:

$$G_{\text{TOTAL,linear}} = \frac{1}{8} \cdot 512 \cdot \frac{1}{16384} = \frac{512}{131072} = \frac{1}{256} \quad (-48.13\,\text{dB})$$

---

## Gain Summary of the DSP Chain

The following table summarizes the gain from the **PFB output** through to the DDS+CIC output:

| Stage | Gain (linear) | Gain (dB) | Description |
|---|---|---|---|
| PFB polyphase accumulation (D=8) | $\times 8$ | +18.06 dB | Coherent sum of 8 polyphase branches |
| PFB qout right-shift (qout=6) | $\div 64 = 2^{-6}$ | −36.12 dB | Bit truncation at PFB output |
| **PFB net** | $\div 8 = 2^{-3}$ | **−18.06 dB** | Combined PFB effect |
| DDS | $\times 1$ | 0 dB | Frequency translation only (ideal) |
| CIC filter (R=8, M=1, N=3) | $\times 512 = 2^{9}$ | +54.19 dB | CIC DC gain |
| qdata scaling (QSEL=14) | $\div 16384 = 2^{-14}$ | −84.29 dB | Bit-slice selection (24→16 bits) |
| **Total (PFB→output)** | $\approx 2^{-8}$ | **−48.13 dB** | Overall DSP chain gain |

---

## Calibration Implication

The calibration constant at the DDS+CIC output is built from all preceding stages:

$$\text{CAL\_CONSTANT\_POST} = -K_{\text{ADC}} - G_{\text{PFB,net}} - G_{\text{CIC}} - G_{\text{QDATA}}$$

$$= 12.89 + 18.06 - 54.19 + 84.29 = 61.05\,\text{dB}$$

This is the **analytical prediction**; the empirical value is obtained from the linear fit to the data (see Section 4).

---


# DDS+CIC into the signal chain

---

### **1. Decimation Handling**

```python
D = 8
if D == 1:
    chain.analysis.set_ddscic_outsel(data="product", cic="no")
else:
    chain.analysis.set_ddscic_outsel(data="product", cic="yes")
    chain.analysis.set_ddscic_decimation(D)
```

* If `D == 1`, the CIC is bypassed, as DDS+CIC requires decimation ≥ 2 to operate.
* For `D > 1`, the CIC is enabled, and the decimation factor is set.

**Note:** CIC filters introduce **amplitude droop** and **phase distortion** depending on decimation factor and number of stages. Keep this in mind when analyzing post-DDC signals.

---

### **2. DDS Frequency**

```python
f_ddscic = fout - FC
chain.analysis.set_ddscic_ddsfreq(f=f_ddscic*1e6)
```

* This sets the DDS frequency relative to your PFB output channel center.
* Make sure the `fout` is within the Nyquist range after decimation: ($ f_{out} < \frac{f_s}{2} $).

---

### **3. Quantization**

```python
chain.analysis.set_ddscic_qprod(14)  # 0-16 bits
```

* This sets the quantization of the output.
* Lower bit width increases quantization noise but reduces resource usage.

---

### **4. Sampling frequency after decimation**

```python
fs_d = chain.fs_ch / chain.analysis.get_ddscic_decimation()
```

* This is the **effective sampling rate** of the post-DDC signal.
* Use `fs_d` for FFT and phase calculations.

---

### **5. Time-domain signal**

```python
[xi,xq] = chain.get_bin_pfb(fout, verbose=True)
x_postddscic_t = xi + 1j*xq
```

* `xi` and `xq` are the in-phase and quadrature components.
* `x_postddscic_t` is the **complex baseband signal** after DDS+CIC.

---

### **6. Saving the data**

```python
np.savetxt(QICK_TOOLS + "/.../x_postddscic_t_data.txt", x_postddscic_t)
```

* Now we have a full time-domain record for later spectrum, amplitude, or phase analysis.

---

### **Next Steps for Analysis**

1. **Amplitude spectrum**

```python
X = np.fft.fftshift(np.fft.fft(x_postddscic_t))
f_axis = np.fft.fftshift(np.fft.fftfreq(len(X), 1/fs_d))
plt.plot(f_axis/1e6, 20*np.log10(np.abs(X)))
plt.xlabel('Frequency [MHz]')
plt.ylabel('Amplitude [dB]')
plt.show()
```

2. **Phase response**

```python
plt.plot(f_axis/1e6, np.angle(X))
plt.xlabel('Frequency [MHz]')
plt.ylabel('Phase [rad]')
plt.show()
```

3. **Check CIC effect**

* If you decimate by `D`, CIC introduces a sinc-shaped amplitude response:
  $$
  H(f) = \left|\frac{\sin(\pi f D / f_s)}{D \sin(\pi f / f_s)}\right|^N
  $$
  where ($N$) = number of CIC stages.

4. **Compare pre- and post-DDC signals** to quantify amplitude droop and phase shift.

---

In [ ]:
%matplotlib ipympl
import numpy as np
import matplotlib.pyplot as plt

# ================================
# User parameters (update these)
# ================================
data_file = "data/f_300-10dBm/x_postddscic_t_data.txt"  # path to your saved data
fs_ch = 256e6       # ADC sampling frequency before decimation (Hz)
D = 8               # CIC decimation factor
N = 3               # Number of CIC stages (3rd order CIC as IP block)
# ================================

# Load complex time-domain data
x_postddscic_t = np.loadtxt(data_file, dtype=complex)

# Effective sampling rate after decimation
fs_d = fs_ch / D
print(f"Effective sampling rate after decimation: {fs_d/1e6:.2f} MHz")

# FFT
X = np.fft.fftshift(np.fft.fft(x_postddscic_t))
f_axis = np.fft.fftshift(np.fft.fftfreq(len(X), 1/fs_d))

# Amplitude in dB
amp_dB = 20*np.log10(np.abs(X)/np.max(np.abs(X)))

# Phase
phase_rad = np.angle(X)

# CIC theoretical amplitude response (normalized)
H_cic = np.sinc(f_axis * D / fs_ch)**N
H_cic_dB = 20*np.log10(H_cic/np.max(H_cic))

# -------------------------------
# Plot amplitude
plt.figure(figsize=(10,5))
plt.plot(f_axis/1e6, amp_dB, label='Measured post-DDSCIC')
plt.plot(f_axis/1e6, H_cic_dB, 'r--', label='CIC theoretical')
plt.xlabel('Frequency [MHz]')
plt.ylabel('Amplitude [dB]')
plt.title(f'Amplitude Spectrum (D={D}, N={N})')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# -------------------------------
# Plot phase
plt.figure(figsize=(10,5))
plt.plot(f_axis/1e6, phase_rad)
plt.xlabel('Frequency [MHz]')
plt.ylabel('Phase [rad]')
plt.title('Phase Spectrum')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ================================
# User parameters
# ================================
data_pre = "data/f_300-10dBm/x_preddscic_t_data.txt"   # signal before DDS+CIC
data_post = "data/f_300-10dBm/x_postddscic_t_data.txt" # signal after DDS+CIC
fs_ch = 256e6       # ADC sampling frequency before decimation
D = 8               # CIC decimation factor
N = 3               # CIC order
# ================================

# Load signals
x_pre = np.loadtxt(data_pre, dtype=complex)
x_post = np.loadtxt(data_post, dtype=complex)

# Effective sampling rate after decimation
fs_d = fs_ch / D
print(f"Effective sampling rate after decimation: {fs_d/1e6:.2f} MHz")

# --- FFT for 'Before' (Full Rate) ---
X_pre = np.fft.fftshift(np.fft.fft(x_pre))
# Use the full ADC sampling rate here
f_axis_pre = np.fft.fftshift(np.fft.fftfreq(len(x_pre), 1/fs_ch))

# --- FFT for 'After' (Decimated Rate) ---
X_post = np.fft.fftshift(np.fft.fft(x_post))
# Use the decimated sampling rate here
f_axis_post = np.fft.fftshift(np.fft.fftfreq(len(x_post), 1/fs_d))

# Normalize amplitudes for comparison
amp_pre_dB = 20*np.log10(np.abs(X_pre)/np.max(np.abs(X_pre)))
amp_post_dB = 20*np.log10(np.abs(X_post)/np.max(np.abs(X_post)))

# Phase
phase_pre_rad = np.angle(X_pre)
phase_post_rad = np.angle(X_post)

# CIC theoretical response
H_cic = np.sinc(f_axis * D / fs_ch)**N
H_cic_dB = 20*np.log10(H_cic/np.max(H_cic))

# -------------------------------
# Plot amplitude comparison
plt.figure(figsize=(10,5))
# Plot 'Before' against the 256MHz-based axis
plt.plot(f_axis_pre/1e6, amp_pre_dB, label='Before DDS+CIC (Full BW)', alpha=0.7)
# Plot 'After' against the 32MHz-based axis
plt.plot(f_axis_post/1e6, amp_post_dB, label='After DDS+CIC (Decimated)', linewidth=2)
#plt.plot(f_axis_pre/1e6, amp_pre_dB, label='Before DDS+CIC')
#plt.plot(f_axis_post/1e6, amp_post_dB, label='After DDS+CIC')
plt.plot(f_axis/1e6, H_cic_dB, 'r--', label='3rd-order CIC theoretical')
plt.xlabel('Frequency [MHz]')
plt.ylabel('Amplitude [dB]')
plt.title('Amplitude Spectrum Comparison')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# -------------------------------
# Plot phase comparison
plt.figure(figsize=(10,5))
plt.plot(f_axis_pre/1e6, phase_pre_rad, label='Before DDS+CIC')
plt.plot(f_axis_post/1e6, phase_post_rad, label='After DDS+CIC')
plt.xlabel('Frequency [MHz]')
plt.ylabel('Phase [rad]')
plt.title('Phase Spectrum Comparison')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ===============================
# LOAD DATA
# ===============================

pre  = np.loadtxt("data/f_300-10dBm/x_preddscic_t_data.txt", dtype=np.complex128)
post = np.loadtxt("data/f_300-10dBm/x_postddscic_t_data.txt", dtype=np.complex128)

Fs_pre  = 256e6
Fs_post = 32e6

# if complex IQ stored as I,Q
#if pre.ndim > 1:
#    pre = pre[:,0] + 1j*pre[:,1]

#if post.ndim > 1:
#    post = post[:,0] + 1j*post[:,1]

# ===============================
# TIME DOMAIN
# ===============================

plt.figure(figsize=(10,4))
plt.plot(np.real(pre[:2000000]), label="pre")
plt.plot(np.real(post[:2000000]), label="post")
plt.legend()
plt.title("Time domain comparison")
plt.show()

# ===============================
# FFT PRE
# ===============================

N = len(pre)

fft_pre = np.fft.fftshift(np.fft.fft(pre))
f_pre = np.fft.fftshift(np.fft.fftfreq(N,1/Fs_pre))

plt.figure(figsize=(7,5))
plt.plot(f_pre/1e6,20*np.log10(np.abs(fft_pre)))
plt.title("FFT before DDS+CIC")
plt.xlabel("MHz")
plt.ylabel("dB")
plt.show()

# ===============================
# FFT POST
# ===============================

N = len(post)

fft_post = np.fft.fftshift(np.fft.fft(post))
f_post = np.fft.fftshift(np.fft.fftfreq(N,1/Fs_post))

plt.figure(figsize=(7,5))
plt.plot(f_post/1e6,20*np.log10(np.abs(fft_post)))
plt.title("FFT after DDS+CIC")
plt.xlabel("MHz")
plt.ylabel("dB")
plt.show()

# ===============================
# AMPLITUDE
# ===============================

amp_pre  = np.abs(pre)
amp_post = np.abs(post)

print("Mean amplitude pre :", np.mean(amp_pre))
print("Mean amplitude post:", np.mean(amp_post))

# ===============================
# PHASE
# ===============================

phase_pre  = np.unwrap(np.angle(pre))
phase_post = np.unwrap(np.angle(post))

plt.figure(figsize=(8,4))
#plt.plot(phase_pre[:10000], label="pre")
plt.plot(phase_post[:60000], label="post")
plt.legend()
plt.title("Phase evolution")
plt.show()

gain_factor = np.mean(np.abs(x_post)) / np.mean(np.abs(x_pre))
print(f"Block gain: {gain_factor:.2f}")

# Tomamos la diferencia de fase entre el último y el primer punto
delta_phase = phase_post[-1] - phase_post[0]
delta_time = len(phase_post) / fs_d

# Calculamos el error de frecuencia en Hz
f_error = delta_phase / (2 * np.pi * delta_time)
print(f"Residual frequency error: {f_error:.2f} Hz")

## ADC Calibration with PFB and DDS+CIC Chain

### Complete Signal Chain
```
Pin_gen → [Analog Atten] → ADC → PFB → DDS → CIC → qdata → Output
```

**Chain breakdown:**
1. **Analog attenuation + ADC:** captured by $K_{\text{ADC}} = -12.89\,\text{dB}$
2. **PFB polyphase accumulation (D=8):** coherent gain $+20\log_{10}(8) = +18.06\,\text{dB}$
3. **PFB qout right-shift (qout=6):** attenuation $-20\log_{10}(2^6) = -36.12\,\text{dB}$  
   → **Net PFB gain:** $G_{\text{PFB,net}} = -18.06\,\text{dB}$
4. **DDS:** frequency translation only (no amplitude gain in ideal case)
5. **CIC filter (R=8, M=1, N=3):** DC gain $G_{\text{CIC}} = (RM)^N = 512 = +54.19\,\text{dB}$
6. **qdata output scaling (QSEL=14):** $G_{\text{QDATA}} = 2^{-14} = -84.29\,\text{dB}$

---

## Calibration Model

The full chain equation (in dB) is:

$$P_{\text{output}}\,(\text{dBFS}) = P_{\text{in}}\,(\text{dBm}) + K_{\text{ADC}} + G_{\text{PFB,net}} + G_{\text{CIC}} + G_{\text{QDATA}}$$

Rearranging to recover RF power:

$$\boxed{P_{\text{in}}\,(\text{dBm}) = P_{\text{output}}\,(\text{dBFS}) + \underbrace{\left(-K_{\text{ADC}} - G_{\text{PFB,net}} - G_{\text{CIC}} - G_{\text{QDATA}}\right)}_{\text{CAL\_CONSTANT\_POST}}}$$

With the values for this experiment:

$$\text{CAL\_CONSTANT\_POST} = -(-12.89) - (-18.06) - (+54.19) - (-84.29) = 61.05\,\text{dB}$$

### Budget summary

| Contribution | Value | Formula |
|---|---|---|
| ADC + analog chain | $K_{\text{ADC}} = -12.89\,\text{dB}$ | from Notebook 01 |
| PFB polyphase accumulation | $+18.06\,\text{dB}$ | $+20\log_{10}(8)$ |
| PFB qout right-shift | $-36.12\,\text{dB}$ | $-20\log_{10}(2^6)$ |
| **Net PFB gain** | $-18.06\,\text{dB}$ | $G_{\text{PFB,net}}$ |
| CIC filter | $+54.19\,\text{dB}$ | $(8 \times 1)^3 = 512$ |
| qdata scaling (QSEL=14) | $-84.29\,\text{dB}$ | $2^{-14}$ |
| **CAL_CONSTANT_POST (analytical)** | **+61.05 dB** | $-K_{\text{ADC}} - G_{\text{PFB,net}} - G_{\text{CIC}} - G_{\text{QDATA}}$ |

> **Note:** This notebook uses `qout = 6`, **not** `qout = 9`. The CIC gain is therefore **not** pre-compensated by the PFB; the full +54.19 dB CIC gain and −18.06 dB PFB net gain both contribute to `CAL_CONSTANT_POST`. This is consistent with [PFB data analysis](./02_PFB_data.ipynb).

---

## Comparison with Notebook 02 (PFB output, DDS+CIC bypassed)

| | PFB output (Notebook 02) | DDS+CIC output (this notebook) |
|---|---|---|
| Signal path | RF → ADC → PFB | RF → ADC → PFB → DDS → CIC → qdata |
| CAL_CONSTANT | +30.95 dB | +61.05 dB (analytical) |
| Difference | — | +30.10 dB = $G_{\text{CIC}} + G_{\text{QDATA}} = 54.19 - 84.29$ wait... |

The DDS+CIC block net effect on CAL_CONSTANT relative to the PFB output is:

$$\Delta\text{CAL} = -G_{\text{CIC}} - G_{\text{QDATA}} = -54.19 + 84.29 = +30.10\,\text{dB}$$

Therefore: $\text{CAL\_CONSTANT\_POST} = \text{CAL\_CONSTANT\_PRE} + 30.10\,\text{dB}$

---

## Effect on Signal Phase

### What does NOT introduce deterministic phase rotation
- `qout` right-shift and `qdata` scaling: amplitude only, phase is preserved.
- CIC filter: introduces a **group delay** $\tau_g = N(R-1)/2 = 10.5$ samples (linear phase), not a phase offset.

### What CAN affect phase
1. **DDS:** introduces a linear phase ramp at $f_{\text{DDS}}$ — does not shift the tone phase after down-conversion to DC.
2. **Acquisition start time:** as in Notebook 02, phases are not reproducible between captures unless RF and ADC clocks share a 10 MHz reference. Large phase spread is expected and normal.

---

## Code: Complete Calibration with DDS+CIC


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob, re

# ============================================================
# SYSTEM PARAMETERS  (consistent with Notebooks 01 and 02)
# ============================================================
FS      = 2**15    # Digital full-scale (12-bit ADC MSB-aligned to 16-bit)
fs_pfb  = 256e6    # PFB channel sample rate [Hz]

# ── ADC calibration (Notebook 01) ───────────────────────────
K_ADC = -12.89     # dB: P_adc(dBFS) = P_in(dBm) + K_ADC

# ── PFB (Notebook 02 corrected model) ───────────────────────
PFB_DECIMATION = 8
PFB_QOUT       = 6

G_polyphase = 20 * np.log10(PFB_DECIMATION)     # +18.06 dB
G_qout      = -20 * np.log10(2**PFB_QOUT)       # -36.12 dB
G_PFB_net   = G_polyphase + G_qout              # -18.06 dB

# Calibration constant at PFB output (from Notebook 02)
CAL_CONSTANT_PRE = -K_ADC - G_PFB_net           # +30.95 dB

# ── CIC ─────────────────────────────────────────────────────
R, M, N_cic = 8, 1, 3
CIC_GAIN     = (R * M)**N_cic                   # 512
CIC_GAIN_DB  = 20 * np.log10(CIC_GAIN)          # +54.19 dB
fs_out       = fs_pfb / R                        # 32 MHz

# ── qdata output scaling ────────────────────────────────────
QSEL_REG          = 14
G_QDATA_DB        = -6.02 * QSEL_REG            # -84.29 dB

# ── DDS+CIC block net effect on calibration constant ────────
#   Delta = -G_CIC - G_QDATA  (both signs relative to PFB output)
DDS_CIC_DELTA_DB  = -CIC_GAIN_DB - G_QDATA_DB   # +30.10 dB

# ── Analytical calibration constant at DDS+CIC output ───────
CAL_CONSTANT_ANALYTICAL = CAL_CONSTANT_PRE + DDS_CIC_DELTA_DB   # +61.05 dB

# ── DDS+CIC internal bit widths ─────────────────────────────
QPROD              = 14   # configured qprod register
QPROD_DDS_INTERNAL = 24   # internal DDS product width (bits)
CIC_OUTPUT_BITS    = 16   # final output width (bits)

tone_bins = 3

print("=== Signal Chain Parameters ===")
print(f"  FS (digital full-scale):               2^15 = {FS}")
print(f"  K_ADC (Notebook 01):                   {K_ADC:.2f} dB")
print()
print("=== PFB Gain Budget (Notebook 02 corrected model) ===")
print(f"  Polyphase accumulation (D=8):          {G_polyphase:+.2f} dB")
print(f"  qout right-shift (qout=6):             {G_qout:+.2f} dB")
print(f"  Net PFB gain G_PFB_net:                {G_PFB_net:+.2f} dB")
print(f"  CAL_CONSTANT_PRE (PFB output):         {CAL_CONSTANT_PRE:.2f} dB")
print()
print("=== DDS+CIC Block ===")
print(f"  CIC gain (R=8, M=1, N=3):              {CIC_GAIN_DB:+.2f} dB")
print(f"  qdata scaling (QSEL={QSEL_REG}):            {G_QDATA_DB:+.2f} dB")
print(f"  DDS+CIC net effect on CAL_CONSTANT:    {DDS_CIC_DELTA_DB:+.2f} dB")
print()
print("=== Calibration at DDS+CIC Output ===")
print(f"  CAL_CONSTANT_ANALYTICAL:               {CAL_CONSTANT_ANALYTICAL:.2f} dB")
print(f"  Formula: P_in (dBm) = P_out (dBFS) + {CAL_CONSTANT_ANALYTICAL:.2f}")
print()

# ============================================================
# LOAD DATA
# ============================================================
data_path = "data/f_300*/x_postddscic_t_data.txt"
files = sorted(glob.glob(data_path),
               key=lambda f: float(re.search(r'(-?\+?\d+)dBm', f).group(1)))

measurements = []

print(f"{'Pin (dBm)':>10}  {'P_out (dBFS)':>13}  {'Phase (°)':>10}  {'SNR (dB)':>9}  {'Peak (%)':>9}")
print("-" * 60)

# ============================================================
# PROCESS FILES
# ============================================================
for file in files:
    Pin_gen   = float(re.search(r'(-?\+?\d+)dBm', file).group(1))
    x         = np.loadtxt(file, dtype=np.complex128)
    N_samples = len(x)
    max_val   = np.max(np.abs(x))
    std_val   = np.std(np.abs(x))

    # Window & FFT
    window        = np.hanning(N_samples)
    coherent_gain = np.sum(window) / N_samples
    xw            = x * window
    fft_complex   = np.fft.fftshift(np.fft.fft(xw) / N_samples)
    freqs         = np.fft.fftshift(np.fft.fftfreq(N_samples, 1 / fs_out))
    mag           = np.abs(fft_complex) / coherent_gain

    # Peak detection & tone power
    peak_idx       = np.argmax(mag)
    f_peak         = freqs[peak_idx]
    idx0, idx1     = peak_idx - tone_bins, peak_idx + tone_bins + 1
    tone_power     = np.sum(mag[idx0:idx1]**2)
    P_output_dbfs  = 10 * np.log10(tone_power / FS**2)
    tone_phase     = np.angle(fft_complex[peak_idx])

    # SNR & SFDR
    noise_mask             = np.ones(len(mag), dtype=bool)
    noise_mask[idx0:idx1]  = False
    noise_power            = np.mean(mag[noise_mask]**2)
    snr_db                 = 10 * np.log10(tone_power / (len(mag) * noise_power))
    mag_no_tone            = mag.copy()
    mag_no_tone[idx0:idx1] = 0
    sfdr_db                = 10 * np.log10(tone_power / np.max(mag_no_tone)**2)

    measurements.append({
        'Pin_gen':       Pin_gen,
        'P_output_dbfs': P_output_dbfs,
        'f_peak':        f_peak,
        'phase':         tone_phase,
        'max_val':       max_val,
        'std_val':       std_val,
        'snr_db':        snr_db,
        'sfdr_db':       sfdr_db,
        'file':          file,
    })

    saturation_pct = (max_val / FS) * 100
    sat_flag = " ⚠️ SAT" if saturation_pct > 90 else ""
    print(f"{Pin_gen:10.1f}  {P_output_dbfs:13.2f}  "
          f"{np.degrees(tone_phase):10.1f}  {snr_db:9.1f}  {saturation_pct:8.1f}%{sat_flag}")

# ============================================================
# EMPIRICAL CALIBRATION — linear fit on linear-region data
# ============================================================
linear_meas = [m for m in measurements if m['Pin_gen'] <= 0]
Pin_vals    = np.array([m['Pin_gen']       for m in linear_meas])
Pout_vals   = np.array([m['P_output_dbfs'] for m in linear_meas])
coeffs_fit  = np.polyfit(Pout_vals, Pin_vals, 1)
slope              = coeffs_fit[0]
CAL_CONSTANT_POST  = coeffs_fit[1]   # empirical calibration constant

print(f"\n=== Empirical linear fit (linear region, Pin ≤ 0 dBm) ===")
print(f"  Pin = {slope:.4f} × P_out + {CAL_CONSTANT_POST:.2f} dB")
print(f"  Expected slope:      1.0000  (actual: {slope:.4f})")
print(f"  Analytical constant: {CAL_CONSTANT_ANALYTICAL:.2f} dB")
print(f"  Empirical constant:  {CAL_CONSTANT_POST:.2f} dB")
print(f"  Agreement:           {abs(CAL_CONSTANT_POST - CAL_CONSTANT_ANALYTICAL):.2f} dB")

# ============================================================
# GAIN BUDGET VERIFICATION
# ============================================================
print(f"\n{'='*70}")
print("GAIN BUDGET VERIFICATION")
print(f"{'='*70}")

DDS_CIC_NET_MEASURED = CAL_CONSTANT_POST - CAL_CONSTANT_PRE

TRUNC_BITS   = QPROD_DDS_INTERNAL - CIC_OUTPUT_BITS
TRUNC_DB     = 6.02 * TRUNC_BITS

print(f"  CAL_CONSTANT_PRE  (PFB output, Notebook 02): {CAL_CONSTANT_PRE:.2f} dB")
print(f"  CAL_CONSTANT_POST (DDS+CIC output, measured): {CAL_CONSTANT_POST:.2f} dB")
print(f"  Measured DDS+CIC delta:   {DDS_CIC_NET_MEASURED:.2f} dB")
print(f"  Analytical DDS+CIC delta: {DDS_CIC_DELTA_DB:.2f} dB")
print(f"    (= -G_CIC - G_QDATA = -{CIC_GAIN_DB:.2f} - ({G_QDATA_DB:.2f}) = {DDS_CIC_DELTA_DB:.2f} dB)")
print(f"  Model vs measurement:     {DDS_CIC_NET_MEASURED - DDS_CIC_DELTA_DB:.2f} dB")

# ============================================================
# PHASE ANALYSIS
# ============================================================
Pin_list   = [m['Pin_gen']           for m in measurements]
phases_deg = [np.degrees(m['phase']) for m in measurements]
phase_mean = np.mean(phases_deg)
phase_std  = np.std(phases_deg)
phase_power_corr = np.corrcoef(Pin_list, phases_deg)[0, 1]

print(f"\n=== Phase Analysis ===")
print(f"  Mean:  {phase_mean:.2f}°")
print(f"  Std:   {phase_std:.2f}°  (large spread is expected — no phase lock between RF gen and ADC clock)")
print(f"  Phase–Power correlation: {phase_power_corr:.3f}")

# ============================================================
# POWER RECOVERY — apply empirical CAL_CONSTANT_POST
# ============================================================
Pin_list      = [m['Pin_gen']       for m in measurements]
Pin_recovered = [m['P_output_dbfs'] + CAL_CONSTANT_POST for m in measurements]
errors        = np.array(Pin_recovered) - np.array(Pin_list)

lin_errors = [e for e, p in zip(errors, Pin_list) if p <= 0]
print(f"\n=== Recovery Accuracy (linear region, Pin ≤ 0 dBm) ===")
print(f"  Mean error:  {np.mean(lin_errors):+.3f} dB")
print(f"  Std dev:      {np.std(lin_errors):.3f} dB")
print(f"  Max |error|:  {np.max(np.abs(lin_errors)):.3f} dB")
print(f"  RMS error:    {np.sqrt(np.mean(np.array(lin_errors)**2)):.3f} dB")

# ============================================================
# PLOTTING
# ============================================================
fig = plt.figure(figsize=(18, 14))
gs  = fig.add_gridspec(4, 3, hspace=0.35, wspace=0.35)
snr_list  = [m['snr_db']  for m in measurements]
sfdr_list = [m['sfdr_db'] for m in measurements]
max_vals  = [m['max_val'] for m in measurements]

# ── Calibration ──────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(Pin_list, Pin_recovered, 'o-', ms=8, lw=2, color='steelblue', label='Recovered')
ax1.plot(Pin_list, Pin_list, 'k--', lw=2, label='Ideal (1:1)')
ax1.set_xlabel('True Input Power (dBm)', fontsize=11)
ax1.set_ylabel('Recovered Power (dBm)', fontsize=11)
ax1.set_title('Post-DDS+CIC Calibration', fontsize=12, fontweight='bold')
ax1.legend(); ax1.grid(True, alpha=0.3)

# ── Error ─────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(Pin_list, errors, 'o-', ms=8, lw=2, color='seagreen')
ax2.axhline(0, color='k', ls='--', lw=2)
ax2.axhline( 0.5, color='orange', ls=':', lw=1.5, alpha=0.8, label='±0.5 dB')
ax2.axhline(-0.5, color='orange', ls=':', lw=1.5, alpha=0.8)
ax2.fill_between(Pin_list, -0.5, 0.5, alpha=0.12, color='seagreen')
ax2.set_xlabel('Input Power (dBm)', fontsize=11)
ax2.set_ylabel('Error (dB)', fontsize=11)
ax2.set_title(f'Calibration Error  (σ={np.std(lin_errors):.3f} dB, linear region)', fontsize=12, fontweight='bold')
ax2.legend(); ax2.grid(True, alpha=0.3)

# ── Phase vs Power ────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
scatter = ax3.scatter(Pin_list, phases_deg, c=snr_list, s=100,
                      cmap='viridis', edgecolors='k', linewidth=1)
ax3.axhline(phase_mean, color='r', ls='--', lw=2, alpha=0.7, label=f'μ={phase_mean:.1f}°')
if abs(phase_power_corr) > 0.5:
    coeffs_phase = np.polyfit(Pin_list, phases_deg, 1)
    p_fit = np.poly1d(coeffs_phase)
    ax3.plot(Pin_list, p_fit(Pin_list), 'r-', lw=2, alpha=0.5,
             label=f'r={phase_power_corr:.2f}')
ax3.legend()
ax3.set_xlabel('Input Power (dBm)', fontsize=11)
ax3.set_ylabel('Phase (degrees)', fontsize=11)
ax3.set_title(f'Phase vs Power (σ={phase_std:.1f}°)', fontsize=12, fontweight='bold')
plt.colorbar(scatter, ax=ax3, label='SNR (dB)')
ax3.grid(True, alpha=0.3)

# ── Saturation Check ──────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(Pin_list, max_vals, 'o-', ms=8, lw=2, color='red')
ax4.axhline(FS, color='k', ls='-', lw=2, label='Full Scale')
ax4.axhline(0.9 * FS, color='orange', ls='--', lw=2, label='90% FS')
ax4.set_xlabel('Input Power (dBm)', fontsize=11)
ax4.set_ylabel('Peak Amplitude', fontsize=11)
ax4.set_title('Saturation Check', fontsize=12, fontweight='bold')
ax4.legend(); ax4.grid(True, alpha=0.3)

# ── SNR ───────────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(Pin_list, snr_list, 'o-', ms=8, lw=2, color='green')
ax5.set_xlabel('Input Power (dBm)', fontsize=11)
ax5.set_ylabel('SNR (dB)', fontsize=11)
ax5.set_title('Signal-to-Noise Ratio', fontsize=12, fontweight='bold')
ax5.grid(True, alpha=0.3)

# ── SFDR ──────────────────────────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
ax6.plot(Pin_list, sfdr_list, 'o-', ms=8, lw=2, color='purple')
ax6.set_xlabel('Input Power (dBm)', fontsize=11)
ax6.set_ylabel('SFDR (dB)', fontsize=11)
ax6.set_title('Spurious-Free Dynamic Range', fontsize=12, fontweight='bold')
ax6.grid(True, alpha=0.3)

# ── Phase Histogram ───────────────────────────────────────────
ax7 = fig.add_subplot(gs[2, 0])
ax7.hist(phases_deg, bins=15, edgecolor='k', alpha=0.7, color='purple')
ax7.axvline(phase_mean, color='r', ls='--', lw=2, label=f'μ={phase_mean:.1f}°')
ax7.axvline(phase_mean - phase_std, color='orange', ls=':', lw=1.5, alpha=0.7)
ax7.axvline(phase_mean + phase_std, color='orange', ls=':', lw=1.5, alpha=0.7)
ax7.set_xlabel('Phase (degrees)', fontsize=11)
ax7.set_ylabel('Count', fontsize=11)
ax7.set_title(f'Phase Distribution (σ={phase_std:.1f}°)', fontsize=12, fontweight='bold')
ax7.legend(); ax7.grid(True, alpha=0.3, axis='y')

# ── Frequency Stability ───────────────────────────────────────
ax8 = fig.add_subplot(gs[2, 1])
freqs_peak = [m['f_peak'] / 1e6 for m in measurements]
ax8.plot(Pin_list, freqs_peak, 'o-', ms=8, lw=2, color='orange')
ax8.axhline(np.mean(freqs_peak), color='r', ls='--', lw=2,
            label=f'μ={np.mean(freqs_peak):.4f} MHz')
ax8.set_xlabel('Input Power (dBm)', fontsize=11)
ax8.set_ylabel('Peak Frequency (MHz)', fontsize=11)
ax8.set_title('Frequency After DDS (residual offset)', fontsize=12, fontweight='bold')
ax8.legend(); ax8.grid(True, alpha=0.3)

# ── IQ Constellation ──────────────────────────────────────────
ax9 = fig.add_subplot(gs[2, 2])
colors_cm = plt.cm.plasma(np.linspace(0, 1, len(measurements)))
for i, (m, c) in enumerate(zip(measurements[::2], colors_cm[::2])):
    x_data    = np.loadtxt(m['file'], dtype=np.complex128)
    subsample = max(1, len(x_data) // 400)
    label     = f"{m['Pin_gen']:+.0f} dBm" if i % 2 == 0 else ""
    ax9.scatter(np.real(x_data[::subsample]), np.imag(x_data[::subsample]),
                alpha=0.3, s=1, color=c, label=label)
ax9.set_xlabel('I (Real)', fontsize=11)
ax9.set_ylabel('Q (Imaginary)', fontsize=11)
ax9.set_title('IQ Constellation', fontsize=12, fontweight='bold')
ax9.legend(fontsize=8, markerscale=5, ncol=2)
ax9.grid(True, alpha=0.3); ax9.axis('equal')

# ── Calibrated Spectrum (mid-power) ───────────────────────────
ax10 = fig.add_subplot(gs[3, :2])
mid_idx    = len(measurements) // 2
x_mid      = np.loadtxt(measurements[mid_idx]['file'], dtype=np.complex128)
window_mid = np.hanning(len(x_mid))
X_mid      = np.fft.fftshift(np.fft.fft(x_mid * window_mid) / len(x_mid))
f_mid      = np.fft.fftshift(np.fft.fftfreq(len(x_mid), 1 / fs_out))
# Amplitude spectrum normalised to FS, then shifted to dBm via empirical CAL_CONSTANT_POST
coherent_gain_mid = np.sum(window_mid) / len(x_mid)
mag_mid_dBm = 20 * np.log10(np.abs(X_mid) / coherent_gain_mid / FS) + CAL_CONSTANT_POST
ax10.plot(f_mid / 1e6, mag_mid_dBm, lw=1, alpha=0.7)
ax10.axvline(0, color='r', ls='--', lw=2, alpha=0.5, label='DC (after DDS)')
ax10.set_xlabel('Frequency (MHz)', fontsize=11)
ax10.set_ylabel('Power (dBm)', fontsize=11)
ax10.set_title(f'Calibrated Spectrum  (Pin={Pin_list[mid_idx]:+.0f} dBm)',
               fontsize=12, fontweight='bold')
ax10.legend(); ax10.grid(True, alpha=0.3)
ax10.set_xlim([-fs_out / 2 / 1e6, fs_out / 2 / 1e6])

# ── Summary box ───────────────────────────────────────────────
ax11 = fig.add_subplot(gs[3, 2])
ax11.axis('off')
summary_text = (
    f"CALIBRATION SUMMARY\n\n"
    f"K_ADC:            {K_ADC:.2f} dB\n"
    f"G_polyphase(D=8): +18.06 dB\n"
    f"G_qout(qout=6):   -36.12 dB\n"
    f"G_PFB_net:        {G_PFB_net:.2f} dB\n"
    f"G_CIC:           +{CIC_GAIN_DB:.2f} dB\n"
    f"G_QDATA:          {G_QDATA_DB:.2f} dB\n\n"
    f"CAL_PRE  (Nb 02): {CAL_CONSTANT_PRE:.2f} dB\n"
    f"CAL_POST (analyt):{CAL_CONSTANT_ANALYTICAL:.2f} dB\n"
    f"CAL_POST (empir): {CAL_CONSTANT_POST:.2f} dB\n"
    f"Agreement:        {abs(CAL_CONSTANT_POST-CAL_CONSTANT_ANALYTICAL):.2f} dB\n\n"
    f"Accuracy:        ±{np.std(lin_errors):.3f} dB\n"
    f"Phase σ:          {phase_std:.2f}°\n"
    f"SNR range:        {np.min(snr_list):.1f}–{np.max(snr_list):.1f} dB"
)
ax11.text(0.05, 0.5, summary_text, transform=ax11.transAxes,
          fontsize=10, verticalalignment='center', fontfamily='monospace',
          bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.savefig('images/ddscic_complete_analysis.pdf', dpi=300, bbox_inches='tight')
plt.show()

# ============================================================
# FINAL REPORT
# ============================================================
print(f"\n{'='*70}")
print("FINAL DIAGNOSTIC REPORT")
print(f"{'='*70}")

print(f"\n GAIN BUDGET:")
print(f"  K_ADC:                     {K_ADC:.2f} dB  (Notebook 01)")
print(f"  G_polyphase (D=8):         {G_polyphase:+.2f} dB  (Notebook 02)")
print(f"  G_qout (qout=6):           {G_qout:+.2f} dB  (Notebook 02)")
print(f"  G_PFB_net:                 {G_PFB_net:+.2f} dB")
print(f"  G_CIC (R=8,M=1,N=3):      {CIC_GAIN_DB:+.2f} dB")
print(f"  G_QDATA (QSEL=14):         {G_QDATA_DB:+.2f} dB")

print(f"\n CALIBRATION:")
print(f"  CAL_CONSTANT_PRE  (Nb 02): {CAL_CONSTANT_PRE:.2f} dB")
print(f"  CAL_CONSTANT_POST (analyt):{CAL_CONSTANT_ANALYTICAL:.2f} dB")
print(f"  CAL_CONSTANT_POST (empir): {CAL_CONSTANT_POST:.2f} dB")
print(f"  Agreement:                 {abs(CAL_CONSTANT_POST-CAL_CONSTANT_ANALYTICAL):.2f} dB")
print(f"  Formula: P_in (dBm) = P_out (dBFS) + {CAL_CONSTANT_POST:.2f}")
print(f"  Linear fit accuracy:      ±{np.std(lin_errors):.3f} dB (linear region)")

print(f"\n PHASE:")
if phase_std < 10:
    print(f"   Excellent phase stability: {phase_std:.2f}° std dev")
elif phase_std < 30:
    print(f"   Good: {phase_std:.2f}° std dev (some variation expected without phase lock)")
elif abs(phase_power_corr) > 0.7:
    print(f"  ! Phase depends on power (r={phase_power_corr:.2f}) — check non-linearity")
else:
    print(f"  ! Large variation ({phase_std:.2f}°) — expected if RF gen and ADC clock are not phase-locked")


## Summary

### Calibration at DDS+CIC Output

The complete calibration chain, using the corrected PFB model from Notebook 02, gives:

$$P_{\text{in}}\,(\text{dBm}) = P_{\text{DDS+CIC}}\,(\text{dBFS}) + \text{CAL\_CONSTANT\_POST}$$

where the analytical prediction is:

$$\text{CAL\_CONSTANT\_POST} = -K_{\text{ADC}} - G_{\text{PFB,net}} - G_{\text{CIC}} - G_{\text{QDATA}}$$

$$= 12.89 + 18.06 - 54.19 + 84.29 = 61.05\,\text{dB}$$

The empirical value is obtained from a linear fit to the linear-region data and should agree with the analytical prediction within ~1 dB.

### Relationship to Notebook 02

| Quantity | PFB output (Nb 02) | DDS+CIC output (this notebook) |
|---|---|---|
| Signal path | RF → ADC → PFB | RF → ADC → PFB → DDS → CIC → qdata |
| $G_{\text{PFB,net}}$ | $-18.06\,\text{dB}$ | $-18.06\,\text{dB}$ (same) |
| Additional stages | — | CIC (+54.19 dB) + qdata (−84.29 dB) |
| CAL_CONSTANT | +30.95 dB | +61.05 dB (analytical) |
| Delta | — | +30.10 dB |

> **Key insight:** the DDS+CIC block adds a net +30.10 dB to the calibration constant relative to the PFB output. This is fully accounted for by $-G_{\text{CIC}} - G_{\text{QDATA}} = -54.19 + 84.29 = +30.10\,\text{dB}$.

### Common mistakes to avoid

1. **Ignoring the polyphase accumulation gain** (+18.06 dB): using only the qout attenuation (−36.12 dB) for `G_PFB_net` introduces a systematic error of ~20 dB. This was identified and corrected in Notebook 02.
2. **Hardcoding CAL_CONSTANT_PRE** instead of deriving it from `K_ADC`, `G_polyphase`, and `G_qout`.
3. **Not including the polyphase gain when computing the CIC pre-compensation condition**: the correct balance requires accounting for the full `G_PFB_net`, not just `G_qout`.
